# Module 04 - Tokenization

Use this notebook as the working space for the Module 04 exercises.

1. Read the lesson page (`docs/modules/04-tokenizer.md`).
2. Open this notebook with `./notebook.sh 04`.
3. Answer the `Question:` / `Answer:` cells below.
4. When you're ready, ask a coding agent to grade your notebook.

Partial work is fine. Blank `Answer: ""` strings are skipped, not counted wrong. If you'd like a hint instead of a grade, write the request inline in the answer string and the agent will tutor first.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt

from g2c.artifacts import find_repo_root
from g2c.tokenizer import BPETokenizer, COURSE_SPECIAL_TOKENS

## Before the Notebook

Use the tests to implement the tokenizer pieces first. The notebook assumes `_get_pair_counts`, `_merge`, `train_step`, `encode`, and `decode` are available as you progress. The scaffolded `train()` loop, Rust-backed `train_fast()`/`encode_fast()`, and tokenizer save/load helpers are implemented for you.

In [ ]:
"Run from the terminal: .venv/bin/python -m pytest tests/test_tokenizer.py -x"
"Question: Which tokenizer test is the next one failing, and what implementation does it point at?"
"Answer: "

## Exercise 1 - Pair Counts and Merge

These two helpers are the BPE algorithm's core data-structure operations. Predict the outputs before running your implementation.

In [ ]:
"Question: What adjacent-pair counts should [1, 2, 1, 2, 3] produce?"
"Answer: "

"Question: Why does a list of length n have only n - 1 adjacent pairs?"
"Answer: "

"Question: What should _merge([1, 1, 1], (1, 1), 99) return, and why?"
"Answer: "

In [ ]:
pair_count_example = [1, 2, 1, 2, 3]
merge_example = [1, 1, 1]

print(BPETokenizer._get_pair_counts(pair_count_example))
print(BPETokenizer._merge(merge_example, (1, 1), 99))

## Exercise 2 - Train BPE on a Tiny Corpus

Train on a very small repeated string first. Call one `train_step()` by hand, then let scaffolded `train()` repeat that step until the target vocabulary size is reached. This makes the learned merges easy to inspect by hand.

In [ ]:
tiny_corpus = "the the the"

"Question: In the initial byte sequence for 'the the the', which adjacent pair should be most frequent?"
"Answer: "

"Question: If the target vocab size is 259, how many merges should be learned from a fresh tokenizer?"
"Answer: "

"Question: Why should the new token IDs start at 256?"
"Answer: "

In [ ]:
step_tok = BPETokenizer()
step_ids = list(tiny_corpus.encode("utf-8"))

print("one step:", step_tok.train_step(step_ids, new_id=len(step_tok.vocab)))
print("merges after one step:", step_tok.merges)
print("learned vocab after one step:", {i: step_tok.vocab[i] for i in range(256, len(step_tok.vocab))})


def print_bpe_progress(info: dict) -> None:
    print(
        f"step {info['steps']} | "
        f"vocab {info['vocab_size']}/{info['target_vocab_size']} | "
        f"last token {info['last_token_repr']} | "
        f"tokens {info['tokens']} | "
        f"last merge count {info['last_merge_count']} | "
        f"elapsed {info['elapsed_seconds']:.3f}s"
    )


tiny_tok = BPETokenizer()
tiny_tok.train(tiny_corpus, vocab_size=259, progress_callback=print_bpe_progress, progress_every=1)
print("merges:", tiny_tok.merges)
print("learned vocab:", {i: tiny_tok.vocab[i] for i in range(256, len(tiny_tok.vocab))})
assert len(tiny_tok.vocab) == 259

## Exercise 3 - Encode, Decode, and Round Trip

A byte-level tokenizer should round-trip any UTF-8 text, including text with characters that were not in the training corpus.

In [ ]:
"Question: Why can a byte-level tokenizer encode text it never saw during training?"
"Answer: "

"Question: During encode, why should the learned merge with the lowest new ID get priority?"
"Answer: "

"Question: What property of vocab entries makes decode lossless?"
"Answer: "

In [ ]:
roundtrip_texts = [
    "the theater there",
    "héllo wörld",
    "unseen text 12345 🌍",
]

roundtrip_tok = BPETokenizer()
roundtrip_tok.train("the quick brown fox jumps over the lazy dog " * 20, vocab_size=300)
for text in roundtrip_texts:
    ids = roundtrip_tok.encode(text)
    decoded = roundtrip_tok.decode(ids)
    print(text, "->", ids, "->", decoded)
    assert decoded == text

## Exercise 4 - Special Tokens as Atomic Markers

Special tokens are reserved control strings. Later modules use them for document boundaries, chat roles, and tool-call events. They should encode as one token ID each, not as ordinary BPE pieces.

In [ ]:
"Question: Why should <|assistant|> be one atomic ID instead of ordinary text pieces?"
"Answer: "

"Question: If a tokenizer reserves 8 special tokens, where should the first learned BPE merge ID start?"
"Answer: "

In [ ]:
special_tok = BPETokenizer.with_course_special_tokens()
chat_text = "<|user|>\nHello<|end|><|assistant|>\nHi!<|end|>"
chat_ids = special_tok.encode(chat_text)

print("course special tokens:")
for token in COURSE_SPECIAL_TOKENS:
    print(token, "->", special_tok.special_to_id[token])

print("\nencoded chat ids:", chat_ids)
assert special_tok.special_to_id["<|user|>"] in chat_ids
assert special_tok.special_to_id["<|assistant|>"] in chat_ids
assert chat_ids.count(special_tok.special_to_id["<|end|>"]) == 2
assert special_tok.decode(chat_ids) == chat_text
assert special_tok.base_vocab_size == 256 + len(COURSE_SPECIAL_TOKENS)

## Exercise 5 - Pre-Tokenization Demo

Production tokenizers (GPT-2, GPT-3, Llama) run a *pre-tokenizer* that splits the input into pieces (words, punctuation, whitespace runs) before BPE pair-counting kicks in. BPE then learns merges only **within** each piece, never across them. This prevents the trainer from inventing tokens like `the<|endoftext|>Once`. The cell below demonstrates the regex split GPT-2 uses.

In [ ]:
"Question: Why might merging across whitespace create less useful tokens?"
"Answer: "

"Question: What would you compare before and after adding pre-tokenization?"
"Answer: "

In [ ]:
import re
from collections import Counter


def simple_pretokenize(text: str) -> list[str]:
    """Split into whitespace spans and non-whitespace spans."""
    return re.findall(r"\s+|\S+", text)


def pair_counts_within_pretokens(text: str) -> Counter[tuple[int, int]]:
    """Count adjacent byte pairs without crossing pre-token boundaries."""
    counts: Counter[tuple[int, int]] = Counter()
    for piece in simple_pretokenize(text):
        counts.update(BPETokenizer._get_pair_counts(list(piece.encode("utf-8"))))
    return counts


pretok_text = "the theater there and the theory"
raw_counts = Counter(BPETokenizer._get_pair_counts(list(pretok_text.encode("utf-8"))))
pretok_counts = pair_counts_within_pretokens(pretok_text)

print("pre-tokens:", simple_pretokenize(pretok_text))
print("raw top pairs:", raw_counts.most_common(8))
print("pre-tokenized top pairs:", pretok_counts.most_common(8))
print("\nNotice that pre-tokenized counts never include a pair crossing from the end of one word into the next span.")

## Train the TinyShakespeare tokenizer

The remaining exercises run on real text. TinyShakespeare (~1 MB) is small enough that we just retrain a tokenizer here every time this cell runs, so there's no "is the artifact stale?" guesswork. The artifact is saved to `artifacts/tokenizers/ShakespeareTokenizer/` for downstream modules to load.

In [ ]:
from g2c.artifacts import (
    TokenizerArtifactConfig,
    train_or_load_tokenizer_artifact,
)
from g2c.notebook_extras.tokenizer import (
    inspect_tokenizer_artifact,
    learned_vocab_window,
    make_artifact_display,
    measure_vocab_compression,
    plot_frequent_token_histogram,
    plot_vocab_divider_histogram,
)

repo_root = find_repo_root()


In [ ]:
shakespeare_config = TokenizerArtifactConfig(
    name="ShakespeareTokenizer",
    source="tinyshakespeare",
    vocab_size=4096,
    max_chars=1_000_000,
    use_fast=True,
    chunk_chars=8_192,
    notes="Reusable Shakespeare tokenizer for compression and inspection exercises.",
)

shakespeare_status = make_artifact_display(shakespeare_config.name)
shakespeare_artifact = train_or_load_tokenizer_artifact(
    shakespeare_config,
    repo_root=repo_root,
    force=True,
    progress_callback=shakespeare_status,
    status_callback=shakespeare_status,
)
print(f"trained vocab: {len(shakespeare_artifact.tokenizer.vocab):,}")
print(f"corpus chars: {len(shakespeare_artifact.text):,}")

## Exercise 6 - Vocab Size vs Compression

The previous cell trained a Shakespeare tokenizer to vocab 4096. Now sweep across smaller effective vocab sizes by truncating that same merge list and measuring how many tokens a fixed Shakespeare passage compresses to. The lesson is the *shape of the curve*, not the loop body — `measure_vocab_compression` is provided.

In [ ]:
"Question: What do you predict will happen to token count as vocab size increases?"
"Answer: "

"Question: Why might the improvement show diminishing returns?"
"Answer: "

"Question: What is the tradeoff of using a very large vocabulary in a real model?"
"Answer: "

In [ ]:
# measure_vocab_compression lives in g2c/notebook_extras/tokenizer.py.
# It is measurement glue for this visualization, not a BPE algorithm step.
measure_vocab_compression


In [ ]:
shakespeare_passage = shakespeare_artifact.text[:2000]
vocab_sizes = [
    shakespeare_artifact.tokenizer.base_vocab_size,
    320,
    512,
    1024,
    2048,
    4096,
]
compression_rows = measure_vocab_compression(
    shakespeare_artifact.tokenizer,
    shakespeare_passage,
    vocab_sizes,
)
for row in compression_rows:
    print(
        f"  vocab={row['vocab_size']:>5} | "
        f"{row['tokens']:>5} tokens | "
        f"{row['chars_per_token']:.2f} chars/token"
    )

plt.figure(figsize=(7, 3.5))
plt.plot(
    [r["vocab_size"] for r in compression_rows],
    [r["tokens"] for r in compression_rows],
    "o-",
)
plt.xlabel("vocab size")
plt.ylabel(f"tokens for {len(shakespeare_passage)}-char passage")
plt.title("Shakespeare passage tokens vs vocab size")
plt.xscale("log")
plt.grid(True, alpha=0.3)
plt.show()

## Exercise 7 - Inspect Learned Tokens

Look at the actual byte sequences BPE learned. Token IDs just past the special-token reserved range should hold the most-frequent short patterns; the highest-numbered learned IDs should hold the longest learned phrases.

In [ ]:
"Question: Why should token IDs just above 255 usually represent very common patterns?"
"Answer: "

"Question: What kind of learned token would count as corpus-specific rather than generally useful?"
"Answer: "

In [ ]:
# learned_vocab_window lives in g2c/notebook_extras/tokenizer.py.
# It is an inspection helper, not part of the BPE algorithm implementation.
learned_vocab_window


In [ ]:
shakespeare_tok = shakespeare_artifact.tokenizer
first_tokens, last_tokens = learned_vocab_window(shakespeare_tok, n=20)

print(f"first {len(first_tokens)} learned tokens:")
for token_id, token_text in first_tokens:
    print(f"  {token_id}: {token_text!r}")

print()
print(f"last {len(last_tokens)} learned tokens:")
for token_id, token_text in last_tokens:
    print(f"  {token_id}: {token_text!r}")


In [ ]:
inspect_tokenizer_artifact(shakespeare_artifact, show_plots=False)


In [ ]:
plot_frequent_token_histogram(shakespeare_artifact)


In [ ]:
plot_vocab_divider_histogram(shakespeare_artifact)


In [ ]:
"Question: Name two early learned tokens and explain why they are plausible frequent patterns."
"Answer: "

"Question: Name one late learned token and explain what makes it more specific."
"Answer: "

## Exercise 8 - TinyStories Tokenizer (optional)

Train or load a tokenizer on 25M characters of TinyStories with vocab 8192. The artifact saves to `artifacts/tokenizers/StoryTokenizer/` for Module 10's StoryLM to load. `./datasets.sh --tiny` or `./datasets.sh --small` may have already prepared it.

**Heads up: a fresh train can take ~10 minutes on an M-series Mac.** The cell loads an existing artifact by default; set `force=True` below only when you intentionally want to retrain.

In [ ]:
tinystories_config = TokenizerArtifactConfig(
    name="StoryTokenizer",
    source="tinystories",
    vocab_size=8192,
    max_chars=25_000_000,
    use_fast=True,
    chunk_chars=8_192,
    notes="TinyStories tokenizer for StoryLM. ~10 minute training run.",
)

tinystories_status = make_artifact_display(tinystories_config.name)
tinystories_artifact = train_or_load_tokenizer_artifact(
    tinystories_config,
    repo_root=repo_root,
    force=False,
    progress_callback=tinystories_status,
    status_callback=tinystories_status,
)


In [ ]:
if tinystories_artifact is not None:
    inspect_tokenizer_artifact(tinystories_artifact, show_plots=False)


In [ ]:
if tinystories_artifact is not None:
    plot_frequent_token_histogram(tinystories_artifact)


In [ ]:
if tinystories_artifact is not None:
    plot_vocab_divider_histogram(tinystories_artifact)


## Exercise 9 - G2C Corpus Tokenizer (optional)

Train or load a broader tokenizer on 100M characters of the mixed G2C corpus with vocab 16384. You'll need to manually run `./datasets.sh --small` or `./datasets.sh` before starting this exercise to initialize the data. (The notebook launcher only calls `./datasets.sh --tiny`.)

**Heads up: a fresh train can take tens of minutes.** The cell loads an existing artifact by default; set `force=True` below only when you intentionally want to retrain.

In [ ]:
g2c_config = TokenizerArtifactConfig(
    name="G2CTokenizer",
    source="g2c",
    vocab_size=16384,
    max_chars=100_000_000,
    use_fast=True,
    chunk_chars=8_192,
    notes="Broad course-corpus tokenizer for TinyLLM-small experiments. ~1 hour training run.",
)

g2c_status = make_artifact_display(g2c_config.name)
g2c_artifact = train_or_load_tokenizer_artifact(
    g2c_config,
    repo_root=repo_root,
    force=False,
    progress_callback=g2c_status,
    status_callback=g2c_status,
)


In [ ]:
if g2c_artifact is not None:
    inspect_tokenizer_artifact(g2c_artifact, show_plots=False)


In [ ]:
if g2c_artifact is not None:
    plot_frequent_token_histogram(g2c_artifact)


In [ ]:
if g2c_artifact is not None:
    plot_vocab_divider_histogram(g2c_artifact)


## Submission Notes

When complete, ask a coding agent to grade your Module 04 notebook. Partial work is fine: the agent should grade answered questions and implemented sections, then skip blank prompts.